# Portion Estimation — Food-101
## Pipeline: CNN → YOLO → SAM-b → Ratio Features → S/M/L

```
Image
  │
  ├─[1]─ EfficientNet-B0 (fine-tuned) ──────► food_class
  │
  ├─[2]─ YOLOv8n ──────────────────────────► bbox (x1,y1,x2,y2)
  │         fallback: largest object → center 70%
  │
  ├─[3]─ SAM-b (box prompt) ───────────────► refined binary mask
  │
  ├─[4]─ Mask cleanup (morphology) ─────────► clean mask
  │
  ├─[5]─ Geometry features (R1–R5) ──────────► ratio vector
  │
  └─[6]─ Class-aware scoring ──────────────► Small / Medium / Large
```

In [ ]:
# ultralytics cung cấp cả YOLOv8 lẫn SAM trong 1 package
!pip install ultralytics timm --quiet

In [ ]:
import os, cv2, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from torchvision import transforms
from PIL import Image
from ultralytics import YOLO, SAM
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)

# ── Paths ──
IMAGE_DIR  = Path('/kaggle/input/datasets/minhquang2701/food101-15cl')
CNN_CKPT   = '/kaggle/input/YOUR_CNN_DATASET/efficientnet_food15.pth'  # <-- đổi path
OUTPUT_DIR = Path('/kaggle/working/portion_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Classes (phải đúng thứ tự khi train CNN) ──
CLASSES = [
    'caesar_salad','chocolate_cake','donuts','dumplings',
    'french_fries','fried_rice','grilled_salmon','hamburger',
    'hot_dog','ice_cream','omelette','pancakes',
    'pizza','ramen','sushi',
]
NUM_CLASSES = len(CLASSES)  # 15

# ── YOLO: COCO food class ids ──
YOLO_FOOD_IDS = {46,47,48,49,50,51,52,53,54,55}
# 46=banana 47=apple 48=sandwich 49=orange 50=broccoli
# 51=carrot 52=hot_dog 53=pizza 54=donut 55=cake

# ── Portion thresholds ──
THRESHOLDS   = {'small': 0.28, 'large': 0.55}
COLOR_MAP    = {'Small':'#3498db','Medium':'#f39c12','Large':'#e74c3c'}
SEED         = 42
N_LABEL_PER_CLASS = 10   # 10 ảnh/class = 150 ảnh tổng
N_SAMPLE_PER_CLASS = 100

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# ════════════════════════════════════════════════
# LOAD 3 MODELS — chạy 1 lần
# ════════════════════════════════════════════════

# ── [A] EfficientNet-B0 (CNN đã fine-tune) ──
cnn_model = timm.create_model('efficientnet_b0', pretrained=False,
                               num_classes=NUM_CLASSES)
cnn_model.load_state_dict(torch.load(CNN_CKPT, map_location=DEVICE))
cnn_model = cnn_model.to(DEVICE).eval()
print('✓ CNN loaded')

# Transform giống lúc train
cnn_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225]),
])

# ── [B] YOLOv8n (general object detector) ──
yolo_model = YOLO('yolov8n.pt')   # auto-download ~6 MB
print('✓ YOLOv8n loaded')

# ── [C] SAM-b ──
sam_model = SAM('sam_b.pt')       # auto-download ~375 MB
sam_model.to(DEVICE)
print('✓ SAM-b loaded')

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 0 — TIỀN XỬ LÝ (dùng lại từ giữa kỳ)
# ════════════════════════════════════════════════
def preprocess_image(img_bgr: np.ndarray, size: int = 512) -> np.ndarray:
    """
    Bilateral → CLAHE (kênh L) → Resize giữ tỷ lệ + pad reflect
    Output: BGR uint8 512×512
    """
    f = cv2.bilateralFilter(img_bgr, 5, 50, 50)
    lab = cv2.cvtColor(f, cv2.COLOR_BGR2LAB)
    lab[:,:,0] = cv2.createCLAHE(2.0,(8,8)).apply(lab[:,:,0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    h, w  = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h*scale), int(w*scale)
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    ph, pw = size-nh, size-nw
    img = cv2.copyMakeBorder(img, ph//2, ph-ph//2,
                              pw//2, pw-pw//2, cv2.BORDER_REFLECT)
    return img

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 1 — CNN CLASSIFY
# Output: tên class + confidence
# ════════════════════════════════════════════════
@torch.no_grad()
def cnn_classify(img_bgr: np.ndarray) -> tuple[str, float]:
    """
    Input : ảnh BGR (bất kỳ kích thước)
    Output: (class_name, confidence)
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    tensor  = cnn_transform(pil_img).unsqueeze(0).to(DEVICE)
    logits  = cnn_model(tensor)                   # (1, 15)
    probs   = torch.softmax(logits, dim=1)[0]     # (15,)
    idx     = int(probs.argmax())
    return CLASSES[idx], float(probs[idx])

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 2 — YOLO DETECT FOOD REGION
# Output: bbox [x1, y1, x2, y2] trong ảnh 512×512
#
# Chiến lược fallback (theo thứ tự ưu tiên):
#   1. Box có class_id thuộc YOLO_FOOD_IDS + conf > 0.3
#   2. Box bất kỳ có conf cao nhất (thức ăn thường là object chính)
#   3. Center 70% — khi YOLO không detect được gì
# ════════════════════════════════════════════════
def yolo_detect(img_bgr: np.ndarray,
                conf_thr: float = 0.25) -> tuple[list, str]:
    """
    Trả về:
      bbox   : [x1, y1, x2, y2] pixel coords
      source : 'food_class' | 'any_object' | 'center_fallback'
    """
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = yolo_model(img_rgb, conf=conf_thr, verbose=False)
    boxes   = results[0].boxes  # Boxes object

    if boxes is not None and len(boxes) > 0:
        xyxy    = boxes.xyxy.cpu().numpy()    # (N,4)
        confs   = boxes.conf.cpu().numpy()    # (N,)
        cls_ids = boxes.cls.cpu().numpy().astype(int)  # (N,)

        # ── Ưu tiên 1: food class ──
        food_mask = np.array([c in YOLO_FOOD_IDS for c in cls_ids])
        if food_mask.any():
            best = int(np.argmax(confs * food_mask.astype(float)))
            return xyxy[best].tolist(), 'food_class'

        # ── Ưu tiên 2: any object (highest conf) ──
        best = int(np.argmax(confs))
        return xyxy[best].tolist(), 'any_object'

    # ── Fallback: center 70% ──
    margin = 0.15
    x1 = int(w * margin);  y1 = int(h * margin)
    x2 = int(w * (1-margin)); y2 = int(h * (1-margin))
    return [x1, y1, x2, y2], 'center_fallback'


def expand_bbox(bbox: list, img_shape: tuple,
                margin: float = 0.05) -> list:
    """
    Mở rộng bbox thêm margin% mỗi phía.
    Đảm bảo SAM có đủ context ở viền thức ăn.
    """
    h, w = img_shape[:2]
    x1, y1, x2, y2 = bbox
    dx = (x2 - x1) * margin;  dy = (y2 - y1) * margin
    return [
        max(0,   int(x1 - dx)),
        max(0,   int(y1 - dy)),
        min(w-1, int(x2 + dx)),
        min(h-1, int(y2 + dy)),
    ]

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 3 — SAM-b REFINE FOOD BOUNDARY
# Dùng bbox từ YOLO làm box prompt → mask chính xác hơn
# ════════════════════════════════════════════════
def sam_refine(img_bgr: np.ndarray, bbox: list) -> np.ndarray:
    """
    Input : ảnh BGR + bbox [x1,y1,x2,y2]
    Output: binary mask uint8 (0/255)
    """
    h, w    = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    try:
        results      = sam_model(img_rgb, bboxes=[bbox], verbose=False)
        masks_tensor = results[0].masks

        if masks_tensor is not None and len(masks_tensor) > 0:
            masks_np = masks_tensor.data.cpu().numpy()  # (N, H, W)
            # Chọn mask có diện tích lớn nhất
            best  = masks_np[int(np.argmax([m.sum() for m in masks_np]))]
            binary = (best * 255).astype(np.uint8)
        else:
            # Fallback: vẽ bbox thành mask
            binary = np.zeros((h, w), dtype=np.uint8)
            x1,y1,x2,y2 = [int(v) for v in bbox]
            binary[y1:y2, x1:x2] = 255

    except Exception as e:
        print(f'  [SAM warning] {e}')
        binary = np.zeros((h, w), dtype=np.uint8)
        x1,y1,x2,y2 = [int(v) for v in bbox]
        binary[y1:y2, x1:x2] = 255

    if binary.shape != (h, w):
        binary = cv2.resize(binary, (w, h),
                            interpolation=cv2.INTER_NEAREST)
    return binary

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 4 — MASK CLEANUP
# ════════════════════════════════════════════════
def mask_cleanup(mask: np.ndarray) -> np.ndarray:
    """
    3 bước cleanup theo thứ tự:
      1. Closing (15×15)  — lấp lỗ hổng, nối đoạn đứt
      2. Opening  (5×5)   — loại noise nhỏ bên ngoài
      3. Giữ connected component lớn nhất — loại vùng rời rạc nhỏ
    """
    # Closing
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15,15))
    mask    = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)

    # Opening
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k_open)

    # Giữ component lớn nhất
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask, connectivity=8)
    if n_labels > 1:
        # stats[0] là background, bỏ qua
        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask = np.where(labels == largest_label,
                        np.uint8(255), np.uint8(0))
    return mask

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 5 — GEOMETRY FEATURE EXTRACTION
# 5 ratio từ mask, tất cả ∈ [0,1]
# ════════════════════════════════════════════════
def extract_ratio_features(img_bgr: np.ndarray,
                            mask: np.ndarray,
                            food_class: str = '') -> dict:
    h, w         = img_bgr.shape[:2]
    total_pixels = h * w
    fg_pixels    = int(mask.sum() // 255)

    feats = {'food_class': food_class,
             'total_pixels': total_pixels,
             'fg_pixels': fg_pixels}

    # R1: diện tích thức ăn / tổng ảnh
    feats['R1_area_ratio'] = round(fg_pixels / total_pixels
                                   if total_pixels > 0 else 0.0, 4)

    # R2: diện tích thức ăn / diện tích bounding box
    # R3: compactness = 4π·A / P²  (1=tròn hoàn hảo, 0=loằng ngoằng)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        lg = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(lg)
        feats['R2_bbox_fill']   = round(
            min(fg_pixels / (bw*bh), 1.0) if bw*bh > 0 else 0.0, 4)
        perim = cv2.arcLength(lg, True)
        feats['R3_compactness'] = round(
            min(4*np.pi*cv2.contourArea(lg)/(perim**2), 1.0)
            if perim > 0 else 0.0, 4)
        feats['_bbox_contour']  = (x, y, bw, bh)
    else:
        feats['R2_bbox_fill']   = 0.0
        feats['R3_compactness'] = 0.0
        feats['_bbox_contour']  = (0, 0, w, h)

    # R4: độ bão hòa màu trung bình trong foreground
    hsv     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    fg_bool = mask > 0
    feats['R4_saturation'] = round(
        float(hsv[:,:,1][fg_bool].mean()) / 255.0
        if fg_bool.any() else 0.0, 4)

    # R5: mật độ cạnh Canny trong foreground
    edges = cv2.Canny(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), 50, 150)
    feats['R5_edge_density'] = round(
        float(cv2.bitwise_and(edges, edges, mask=mask).sum()//255)
        / fg_pixels if fg_pixels > 0 else 0.0, 4)

    return feats

In [ ]:
# ════════════════════════════════════════════════
# BƯỚC 6 — CLASS-AWARE SCORING
# ════════════════════════════════════════════════

# Trọng số 5 ratio
WEIGHTS = {
    'R1_area_ratio' : 0.50,   # tín hiệu chính
    'R2_bbox_fill'  : 0.25,   # mật độ trong vùng
    'R3_compactness': 0.05,   # hình dạng
    'R4_saturation' : 0.10,   # màu sắc
    'R5_edge_density': 0.10,  # texture
}

# Per-class offset: hiệu chỉnh ngưỡng theo đặc thù từng class
# Giá trị + → nâng ngưỡng (class hay bị ước lượng quá cao)
# Giá trị - → hạ ngưỡng  (class hay bị ước lượng quá thấp)
CLASS_OFFSET = {
    'ramen'        :  0.08,  # tô ramen thường chụp gần, chiếm nhiều frame
    'pizza'        :  0.06,  # pizza nguyên chiếc thường chiếm gần hết ảnh
    'fried_rice'   :  0.05,  # bát cơm rang thường chụp từ trên
    'pancakes'     :  0.04,  # chồng pancakes trông nhiều hơn thực tế
    'caesar_salad' : -0.05,  # salad thưa, trông ít hơn thực tế
    'sushi'        : -0.03,  # sushi sắp hàng thường chiếm ít diện tích
    'dumplings'    : -0.03,  # dumplings nhỏ gọn
}


def compute_weighted_score(feats: dict) -> float:
    return round(sum(WEIGHTS[k] * feats.get(k, 0.0)
                     for k in WEIGHTS), 4)


def classify_portion(score: float, food_class: str = '') -> str:
    offset  = CLASS_OFFSET.get(food_class, 0.0)
    t_s = THRESHOLDS['small'] + offset
    t_l = THRESHOLDS['large'] + offset
    if score < t_s: return 'Small'
    if score > t_l: return 'Large'
    return 'Medium'

In [ ]:
# ════════════════════════════════════════════════
# FULL PIPELINE — 1 ảnh
# ════════════════════════════════════════════════
def run_pipeline(img_bgr: np.ndarray,
                 food_class_override: str = None) -> dict:
    """
    Input : ảnh BGR + (tùy chọn) override food class
    Output: dict đầy đủ kết quả 6 bước
    """
    # ── 0. Preprocess ──
    pre = preprocess_image(img_bgr, size=512)

    # ── 1. CNN classify ──
    food_class, cnn_conf = cnn_classify(pre)
    if food_class_override:
        food_class = food_class_override   # dùng khi biết nhãn thật

    # ── 2. YOLO detect ──
    raw_bbox, yolo_src = yolo_detect(pre)
    bbox = expand_bbox(raw_bbox, pre.shape, margin=0.05)

    # ── 3. SAM refine ──
    mask_raw = sam_refine(pre, bbox)

    # ── 4. Mask cleanup ──
    mask = mask_cleanup(mask_raw)

    # ── 5. Ratio features ──
    feats = extract_ratio_features(pre, mask, food_class)

    # ── 6. Class-aware scoring ──
    score   = compute_weighted_score(feats)
    portion = classify_portion(score, food_class)

    return {
        **feats,
        'cnn_class'      : food_class,
        'cnn_conf'       : round(cnn_conf, 4),
        'yolo_bbox'      : bbox,
        'yolo_source'    : yolo_src,
        'weighted_score' : score,
        'portion_label'  : portion,
        # internal — không lưu CSV
        '_pre'           : pre,
        '_mask_raw'      : mask_raw,
        '_mask'          : mask,
    }

In [ ]:
# ════════════════════════════════════════════════
# DEMO — kiểm tra pipeline trên 1 ảnh trước khi batch
# ════════════════════════════════════════════════
def visualize_pipeline(result: dict, title: str = '') -> None:
    pre       = result['_pre']
    mask_raw  = result['_mask_raw']
    mask      = result['_mask']
    bbox      = result['yolo_bbox']
    portion   = result['portion_label']
    score     = result['weighted_score']

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    # Panel 1: original + YOLO bbox
    vis = pre.copy()
    x1,y1,x2,y2 = [int(v) for v in bbox]
    cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,0), 3)
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(
        f'[1] CNN: {result["cnn_class"]} ({result["cnn_conf"]:.1%})\n'
        f'[2] YOLO: {result["yolo_source"]}')
    axes[0].axis('off')

    # Panel 2: SAM raw mask
    axes[1].imshow(mask_raw, cmap='gray')
    axes[1].set_title('[3] SAM raw mask')
    axes[1].axis('off')

    # Panel 3: cleaned mask overlay
    overlay = pre.copy()
    overlay[mask == 0] = (overlay[mask == 0] * 0.25).astype(np.uint8)
    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title(
        f'[4] Cleaned mask\nArea: {result["R1_area_ratio"]:.2%}')
    axes[2].axis('off')

    # Panel 4: ratio bars
    names  = ['R1 Area','R2 BBox','R3 Compact','R4 Sat','R5 Edges']
    values = [result['R1_area_ratio'], result['R2_bbox_fill'],
              result['R3_compactness'], result['R4_saturation'],
              result['R5_edge_density']]
    color  = COLOR_MAP[portion]
    axes[3].barh(names, values, color=color, alpha=0.85)
    axes[3].set_xlim(0, 1)
    axes[3].axvline(THRESHOLDS['small'], color='gray',
                    linestyle='--', alpha=0.6)
    axes[3].axvline(THRESHOLDS['large'], color='black',
                    linestyle='--', alpha=0.6)
    axes[3].set_title(
        f'[5][6] Portion: {portion}\nscore={score:.3f}')

    fig.suptitle(title or result['cnn_class'],
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'demo_pipeline.png',
                dpi=150, bbox_inches='tight')
    plt.show()


# Chạy thử trên 1 ảnh pizza
demo_path = sorted((IMAGE_DIR / 'pizza').glob('*.jpg'))[0]
demo_bgr  = cv2.imread(str(demo_path))
demo_res  = run_pipeline(demo_bgr)
visualize_pipeline(demo_res, title='Demo: pizza')

In [ ]:
# ════════════════════════════════════════════════
# BATCH RUN — xem phân bố prediction tổng thể
# ════════════════════════════════════════════════
def collect_paths(classes, n, seed=42):
    random.seed(seed)
    recs = []
    for cls in classes:
        imgs = sorted((IMAGE_DIR / cls).glob('*.jpg'))
        recs += [{'path': str(p), 'food_class': cls}
                 for p in random.sample(imgs, min(n, len(imgs)))]
    return recs


def batch_run(classes, n=N_SAMPLE_PER_CLASS):
    recs    = collect_paths(classes, n, SEED)
    results = []
    yolo_stats = {'food_class':0, 'any_object':0, 'center_fallback':0}

    for rec in tqdm(recs, desc='Batch pipeline'):
        img = cv2.imread(rec['path'])
        if img is None: continue
        r = run_pipeline(img, food_class_override=rec['food_class'])
        yolo_stats[r['yolo_source']] += 1
        results.append({k: v for k, v in r.items()
                        if not k.startswith('_')})

    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_DIR / 'batch_predictions.csv', index=False)

    print(f'\n✓ Đã xử lý {len(df)} ảnh')
    print('\n── YOLO detection source ──')
    for src, cnt in yolo_stats.items():
        print(f'  {src:20s}: {cnt:4d}  ({cnt/len(df):.1%})')
    print('\n── Phân bố portion labels ──')
    print(df['portion_label'].value_counts().to_string())
    print('\n── Phân bố theo class ──')
    print(df.groupby('food_class')['portion_label']
            .value_counts().unstack(fill_value=0).to_string())
    return df


df_batch = batch_run(CLASSES)

In [ ]:
# ════════════════════════════════════════════════
# GÁN NHÃN THỦ CÔNG — 10 ảnh/class = 150 ảnh
#
# Tiêu chí nhất quán:
#   Small  : thức ăn < 1/3 khung hình
#   Medium : thức ăn 1/3 – 2/3 khung hình
#   Large  : thức ăn > 2/3 khung hình hoặc cả đĩa/tô đầy
# ════════════════════════════════════════════════
def interactive_label(recs, save_path):
    results = []
    for i, rec in enumerate(recs):
        img = cv2.imread(rec['path'])
        if img is None: continue
        r    = run_pipeline(img, food_class_override=rec['food_class'])
        pred = r['portion_label']

        fig, axes = plt.subplots(1, 4, figsize=(17, 4))

        # Panel 1: original + bbox
        vis = r['_pre'].copy()
        x1,y1,x2,y2 = [int(v) for v in r['yolo_bbox']]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),3)
        axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f'YOLO: {r["yolo_source"]}')
        axes[0].axis('off')

        # Panel 2: cleaned mask
        overlay = r['_pre'].copy()
        overlay[r['_mask']==0] = (overlay[r['_mask']==0]*0.25).astype(np.uint8)
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'SAM mask\nArea: {r["R1_area_ratio"]:.2%}')
        axes[1].axis('off')

        # Panel 3: ratio bars
        names  = ['R1 Area','R2 BBox','R3 Compact','R4 Sat','R5 Edges']
        values = [r['R1_area_ratio'], r['R2_bbox_fill'],
                  r['R3_compactness'], r['R4_saturation'],
                  r['R5_edge_density']]
        axes[2].barh(names, values, color=COLOR_MAP[pred], alpha=0.85)
        axes[2].set_xlim(0,1)
        axes[2].set_title(f'System: {pred}\nscore={r["weighted_score"]:.3f}')
        axes[2].axvline(THRESHOLDS['small'],color='gray',linestyle='--')
        axes[2].axvline(THRESHOLDS['large'],color='black',linestyle='--')

        # Panel 4: CNN info
        axes[3].text(0.5, 0.6,
            f'CNN: {r["cnn_class"]}\n({r["cnn_conf"]:.1%})',
            ha='center', va='center', fontsize=13,
            transform=axes[3].transAxes,
            bbox=dict(boxstyle='round', facecolor='#EAF2FB', alpha=0.8))
        axes[3].text(0.5, 0.25,
            f'Score: {r["weighted_score"]:.3f}',
            ha='center', va='center', fontsize=11,
            color=COLOR_MAP[pred], fontweight='bold',
            transform=axes[3].transAxes)
        axes[3].axis('off')

        fig.suptitle(
            f'[{i+1}/{len(recs)}] {rec["food_class"]}',
            fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

        while True:
            ans = input(
                f'  System: [{pred}] | '
                'Nhãn (S/M/L hoặc Enter đồng ý): '
            ).strip().upper()
            if ans == '':  label = pred; break
            if ans in ('S','M','L'):
                label = {'S':'Small','M':'Medium','L':'Large'}[ans]
                break
            print('  → Chỉ nhập S, M, L hoặc Enter')

        results.append({
            **{k: v for k, v in r.items() if not k.startswith('_')},
            'manual_label': label,
            'path': rec['path'],
        })
        plt.close('all')

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    print(f'\n✓ Đã lưu {len(df)} nhãn → {save_path}')
    return df


label_csv = OUTPUT_DIR / 'manual_labels.csv'
if not label_csv.exists():
    recs_to_label = collect_paths(CLASSES, N_LABEL_PER_CLASS, SEED)
    df_labeled    = interactive_label(recs_to_label, label_csv)
else:
    print(f'✓ Đã có nhãn: {label_csv}')
    df_labeled = pd.read_csv(label_csv)

In [ ]:
# ════════════════════════════════════════════════
# ĐÁNH GIÁ ĐỊNH LƯỢNG
# ════════════════════════════════════════════════
def evaluate(df):
    y_true = df['manual_label'].tolist()
    y_pred = df['portion_label'].tolist()
    labels = ['Small','Medium','Large']
    acc    = sum(t==p for t,p in zip(y_true,y_pred)) / len(y_true)

    print('='*55)
    print('  PORTION ESTIMATION — KẾT QUẢ ĐÁNH GIÁ')
    print('='*55)
    print(f'  Pipeline : CNN(EfficientNet-B0) → YOLO → SAM-b')
    print(f'  Ảnh      : {len(df)}  ({N_LABEL_PER_CLASS}/class)')
    print(f'  Accuracy : {acc:.1%}')
    print()
    print(classification_report(y_true, y_pred,
                                labels=labels, zero_division=0))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix')

    # Per-class accuracy
    cls_acc = {}
    for cls in CLASSES:
        sub = df[df['food_class']==cls]
        if len(sub): cls_acc[cls] = (sub['manual_label']==sub['portion_label']).mean()
    s_cls = sorted(cls_acc, key=cls_acc.get)
    colors = ['#e74c3c' if cls_acc[c]<0.5 else
              '#f39c12' if cls_acc[c]<0.7 else
              '#27ae60' for c in s_cls]
    axes[1].barh(s_cls, [cls_acc[c] for c in s_cls], color=colors)
    axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='50%')
    axes[1].axvline(acc, color='navy', linestyle='-', alpha=0.7,
                   label=f'Overall {acc:.1%}')
    axes[1].set_xlim(0,1); axes[1].set_xlabel('Accuracy')
    axes[1].set_title('Accuracy theo class'); axes[1].legend()

    # Score distribution
    for lbl in ['Small','Medium','Large']:
        sub = df[df['manual_label']==lbl]['weighted_score']
        axes[2].hist(sub, bins=12, alpha=0.6,
                     label=lbl, color=COLOR_MAP[lbl])
    axes[2].axvline(THRESHOLDS['small'],color='gray',
                    linestyle='--', label='T_small')
    axes[2].axvline(THRESHOLDS['large'],color='black',
                    linestyle='--', label='T_large')
    axes[2].set_xlabel('Weighted Score')
    axes[2].set_title('Score Distribution')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Lỗi thường gặp
    errors = df[df['manual_label']!=df['portion_label']]
    print('\n── Nhầm lẫn phổ biến ──')
    print(errors.groupby(['manual_label','portion_label'])
                .size().reset_index(name='count')
                .sort_values('count',ascending=False)
                .to_string(index=False))

    # YOLO source stats
    if 'yolo_source' in df.columns:
        print('\n── YOLO detection source ──')
        print(df['yolo_source'].value_counts().to_string())


evaluate(df_labeled)

In [ ]:
# ════════════════════════════════════════════════
# VISUALISATION — samples + ratio boxplot
# ════════════════════════════════════════════════
def visualize_samples(df, n_per_portion=2):
    samples = pd.concat([
        df[df['portion_label']==p].sample(
            min(n_per_portion, len(df[df['portion_label']==p])),
            random_state=SEED)
        for p in ['Small','Medium','Large']
    ]).reset_index(drop=True)

    n = len(samples)
    fig, axes = plt.subplots(n, 4, figsize=(17, 4*n))
    if n==1: axes=[axes]

    for i, row in samples.iterrows():
        img = cv2.imread(row['path'])
        r   = run_pipeline(img, food_class_override=row['food_class'])

        vis = r['_pre'].copy()
        x1,y1,x2,y2 = [int(v) for v in r['yolo_bbox']]
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),3)
        axes[i][0].imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(f'{row["food_class"]}\nYOLO: {r["yolo_source"]}')
        axes[i][0].axis('off')

        axes[i][1].imshow(r['_mask_raw'], cmap='gray')
        axes[i][1].set_title('SAM raw'); axes[i][1].axis('off')

        ov = r['_pre'].copy()
        ov[r['_mask']==0]=(ov[r['_mask']==0]*0.25).astype(np.uint8)
        axes[i][2].imshow(cv2.cvtColor(ov,cv2.COLOR_BGR2RGB))
        axes[i][2].set_title(
            f'Cleaned mask\nArea: {r["R1_area_ratio"]:.2%}')
        axes[i][2].axis('off')

        names  = ['R1','R2','R3','R4','R5']
        values = [r['R1_area_ratio'],r['R2_bbox_fill'],
                  r['R3_compactness'],r['R4_saturation'],
                  r['R5_edge_density']]
        color  = COLOR_MAP[r['portion_label']]
        axes[i][3].barh(names, values, color=color, alpha=0.85)
        axes[i][3].set_xlim(0,1)
        axes[i][3].set_title(
            f'{r["portion_label"]}  score={r["weighted_score"]:.3f}')
        for t in THRESHOLDS.values():
            axes[i][3].axvline(t,color='gray',linestyle='--',alpha=0.5)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'portion_samples.png',
                dpi=150, bbox_inches='tight')
    plt.show()


def plot_ratio_boxplots(df):
    cols   = ['R1_area_ratio','R2_bbox_fill','R3_compactness',
              'R4_saturation','R5_edge_density']
    labels = ['R1 Area','R2 BBox','R3 Compact','R4 Sat','R5 Edges']
    fig, axes = plt.subplots(1, 5, figsize=(18, 5))
    for ax, col, lbl in zip(axes, cols, labels):
        data = [df[df['manual_label']==p][col].dropna().values
                for p in ['Small','Medium','Large']]
        bp   = ax.boxplot(data, labels=['S','M','L'], patch_artist=True)
        for patch, c in zip(bp['boxes'],
                            [COLOR_MAP['Small'],COLOR_MAP['Medium'],
                             COLOR_MAP['Large']]):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        ax.set_title(lbl); ax.set_ylim(0,1)
    fig.suptitle('Phân bố Ratio theo nhãn thật — '
                 'hộp tách xa = ratio đóng góp nhiều',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'ratio_boxplots.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('→ Ratio tách tốt nhất: tăng WEIGHTS tương ứng để cải thiện accuracy')


visualize_samples(df_labeled, n_per_portion=2)
plot_ratio_boxplots(df_labeled)
print(f'\n✓ Xong! Kết quả: {OUTPUT_DIR}')